# 🚀 دفتر `main` — NIG-TimeNet (PyTorch): من البيانات إلى الربح بعد التكلفة

**هذا الدفتر «مُشغِّل» لا أكثر: لا منطق فيه (القاعدة G6).** كل ما يفعله يعيش في وحدات الحزمة `src/nig_timenet_torch/` ولها اختبارات؛ الخلايا هنا تستدعي
دوالها وتعرض نتائجها لتقرأها قبل الانتقال إلى الخطوة التالية. تعدّل **خلية واحدة فقط** (الخلية الثالثة، «الإعدادات»)، ثم `Runtime ▸ Run all`.

### ماذا يفعل بالترتيب

| # | القسم | ما يفعله | الوحدة التي تنفّذه |
|---|---|---|---|
| ١ | الإعدادات | كل خيارات التشغيل في مكان واحد، لكل حقل تعليق: معناه، قيمه المسموحة، افتراضيه، أثره | — (قيم فقط) |
| ٢ | الكود | يختار مصدر الكود: **GitHub** أو **نسخة المشروع على Drive**، ويحدّث نسخة Drive من GitHub عند الطلب دون أن يمحو تعديلاتك | `colab/bootstrap.py` |
| ٣ | البيئة | بيئة بايثون مخزّنة على Drive (tar واحد) فلا تُثبَّت الحزم من الصفر كل جلسة | `colab/env_cache.py` |
| ٤ | الجهاز | يتحقق من GPU ويسجّل `setup.json` (القاعدة E1) | `colab/session.py` |
| ٥ | مجلد التشغيل | يحدّد مكان العمل (Drive أو قرص الجهاز) ويستعيد تشغيلاً سابقاً من Drive | `colab/drive_sync.py` |
| ٦ | البيانات والتقسيم | يحمّل «العقد» (artifact) ويقسّمه زمنياً (train / val / test) مع عزل | `workflow/run.py::prepare` |
| ٧ | النموذج | يبني النموذج والمدرّب وهوية التشغيل | `workflow/run.py::build` |
| ٨ | التدريب | يدرّب ويستأنف من آخر نقطة حفظ في مجلد التشغيل | `workflow/run.py::train` |
| ٩ | تقرير الصحة | فحص الطبقات والرؤوس على عيّنة من val | `workflow/run.py::health_step` |
| ١٠ | تقرير الاختبار | مقاييس test: rank IC، التغطية، T-sharp، T-conf | `workflow/run.py::test_step` |
| ١١ | بطارية التقييم | الربح بعد التكلفة (T-profit) مع خطوط الأساس وفاصل الثقة | `workflow/battery_run.py` |
| ١٢ | الملخّص والدفع إلى Drive | `baseline_summary.json` ثم نسخ مجلد التشغيل إلى Drive | `workflow/run.py`, `colab/drive_sync.py` |
| ١٣ | بطارية مجلد منتهٍ | تعيد البطارية من جداول مجلد تشغيل سابق (تكلفة أخرى أو متغيّرات أكثر) دون نموذج | `workflow/battery_run.py` |
| ١٤ | ما ليس هنا بعد | أقسام الدفتر القديم (TensorFlow) التي لا مقابل لها حالياً، ومتى ستأتي | — |

### الهدف الذي يُقاس به كل شيء هنا
قرار المالك (2026-11-07): الهدف هو **T-profit = الربح بعد التكلفة** (0.0008 لكل جهة، أي 0.0016 للصفقة الكاملة). T-skill وT-conf وT-sharp وسائل لا غايات.
اتجاه رأسَي high/low معلَّق؛ T-sharp لرأس close **معلوماتي فقط** (O-1). لا تُبنى قراءة «النموذج جيد» على رقم دون خط أساسه (القاعدة C1).

### ما الذي يُخرجه وأين
* **مجلد التدريب** `…/<RUN_NAME>/train`: نقاط الحفظ (تُستأنف منها الحقب)، تقرير الصحة. **مجلد المخرجات** `…/<RUN_NAME>/out`: `baseline_summary.json`، `test_report.json`،
  `health_report.md`، `battery_report.txt`، `battery_profit.json`، `battery_table_{val,test}.parquet`.
* افتراضياً يعمل التشغيل **على Drive مباشرة** (`WORK_ON = "drive"`) فلا يضيع شيء إن انقطعت الجلسة؛ مع `WORK_ON = "local"` يعمل على قرص الجهاز (أسرع) ثم يُنسخ إلى Drive في القسم ١٢.

### كم يستغرق؟ (على T4)
* تشغيل كامل بخمس حقب على كامل البيانات: نحو **16 دقيقة** (المرجع: `baseline-001-s0-r2`، 972 ثانية؛ الحقبة الأولى هي الأفضل، ثم يبدأ التحقق بالتدهور — انظر `docs/STATUS.md`).
* تشغيل تجريبي سريع (smoke): `"split.max_rows": 20000` و`"train.epochs": 1` ⇒ بضع دقائق، يكفي للتأكد أن كل شيء يعمل قبل تشغيل طويل (القاعدة E5).
* إعداد الجلسة: دقيقتان إلى ثلاث أول مرة (بناء البيئة)، ثم أقل من دقيقة بفضل البيئة المخزّنة.

### دفاتر أخرى
* **`colab_nig_timenet_torch.ipynb`**: المُشغِّل المؤتمت (طابور الوظائف ← تشغيل ← دفع النتائج إلى فرع `colab-results`). هذا الدفتر (`main`) تفاعلي: ترى كل خطوة وتتحكم بها.
* **`data_pipeline.ipynb`**: يبني ملف البيانات (artifact) من الشموع الخام. يجب تشغيله مرة قبل هذا الدفتر.

[افتح هذا الدفتر في Colab](https://colab.research.google.com/github/yuosef772424/nig-timenet-torch/blob/main/notebooks/main.ipynb)

## ٠) وصفات سريعة — ماذا أعدّل لكل حالة؟

| أريد أن… | عدّل في خلية الإعدادات | ملاحظة |
|---|---|---|
| **تجربة سريعة** للتأكد أن كل شيء يعمل | `"split.max_rows": 20000`، `"train.epochs": 1`، `RUN_NAME = "smoke-1"` | بضع دقائق؛ أرقامها ليست نتيجة (القاعدة E5: شغّلها قبل أي تشغيل طويل على commit جديد) |
| **تشغيل مرجعي كامل** | الافتراضيات كما هي، `RUN_NAME = "run-001"` | نحو 16 دقيقة على T4 |
| **إكمال تشغيل انقطع** | نفس `RUN_NAME` تماماً وشغّل الدفتر من أوله | يكمل من آخر نقطة حفظ في `train/`؛ يرفض إن تغيّرت الإعدادات |
| **البدء من تشغيل سابق** | `RESTORE_FROM = "<اسم تشغيل على Drive>"` و`RUN_NAME` جديد | يُنسخ المجلد قبل أي خطوة؛ لا يستبدل ملفاً مختلفاً إلا مع `OVERWRITE_ON_RESTORE` |
| **العمل بلا إنترنت** (نسخة Drive موجودة) | `REPO_SOURCE = "drive"` | لا شبكة ولا توكن |
| **تحديث نسخة Drive إلى آخر `main`** | `REPO_SOURCE = "github"`، `UPDATE_DRIVE_COPY = True` | يرفض إن عدّلتَ ملفات فيها (`DISCARD_LOCAL_CHANGES` للتجاهل) |
| **تثبيت الكود على commit محدّد** | `TORCH_REF = "<SHA كامل>"` | يُسجَّل في هوية التشغيل |
| **سرعة أعلى للقراءة والكتابة** | `WORK_ON = "local"` | تأكّد أن `PUSH_TO_DRIVE = True`؛ لو انقطعت الجلسة قبل النسخ ضاع التشغيل |
| **قراءة الربح بتكلفة أخرى** | `BATTERY_FOLDER = "<مجلد out لتشغيل منتهٍ>"`، `BATTERY_COST = 0.0012` | القسم ١٣، بلا نموذج |
| **مقارنة ذراعين** | شغّلهما بنفس الـ commit وبذرتين على الأقل (`train.seed`) | بذرة واحدة «نظرة» لا نتيجة (القاعدة C4) |
| **تجربة إصلاحات FX1** (D-022) | `FX1_BUNDLE = True` (وللأفضل في دراسة E1 أيضاً `FX1_NIG_SCALE = 0.01`)، و`RUN_NAME` جديد | مفاتيح النموذج `bn_warmup` و`order_detach_close` و`nig_scale`؛ جُرّبت على بيانات اصطناعية وبـGRU فقط |
| **أوزان آخر حقبة بدل أفضلها** | `KEEP_LAST_WEIGHTS = True`، و`RUN_NAME` جديد | تقارير التشغيل كلها على أوزان آخر حقبة؛ `best.pt` يُحفظ أيضاً |
| **قراءة تشغيل منتهٍ بأوزانه الأخيرة** | `RESTORE_FROM = "<تشغيل منتهٍ>"`، `RUN_NAME` جديد، `LOAD_WEIGHTS = "last"` (أو `"best"`) | بلا تدريب؛ الإعدادات يجب أن تطابق التشغيل الأصلي، وبلا `KEEP_LAST_WEIGHTS` |
| **حذف عملات من التدريب والتحقق والاختبار** | `"split.exclude_coins": ["BTCUSDT", "ETH"]` في `CHANGES`، و`RUN_NAME` جديد | بعد التقسيم الزمني فلا تتحرك التواريخ؛ اسم غير موجود = خطأ؛ تغييره دون تغيير `RUN_NAME` يكمل بصمت على صفوف مختلفة |


## ١) الإعدادات — كل ما تعدّله هنا (الخلية التالية)

الخلية التالية **الوحيدة** التي تُعدَّل. ثلاث مجموعات:

1. **الكود والبيئة والمسارات:** من أين يأتي الكود، أين نسخة Drive، أين تُخزَّن البيئة، اسم التشغيل.
2. **`CHANGES`:** *كل* حقل من `RunSettings` (الكائن الصارم الذي تأخذه خطوات `workflow/run.py`) بقيمته الافتراضية الحالية وتعليق عربي: ما معناه، ما القيم المسموحة، وما أثره.
   لتجربة تغيير عدّل القيمة هنا فقط. **اسم حقل خاطئ أو قيمة غير صالحة = خطأ فوري** مع اقتراح الاسم الصحيح، لا تجاهل صامت (القاعدة B5).
3. **البطارية:** تكلفة الجهة وعدد المتغيّرات لقسم «بطارية مجلد منتهٍ».

القيم المكتوبة هي **الافتراضيات الحالية حرفياً** (اختبار يقارنها بالكود فيكشف أي قيمة قديمة): تشغيل الدفتر بلا تعديل يعطي نفس سلوك `jobs/baseline.py`.
اختبار آخر يتأكد أن `CHANGES` يغطي كل حقل في `RunSettings` — إن أُضيف حقل جديد في الكود ولم يُضَف هنا يفشل الاختبار.

> **تشغيل سريع للتجربة:** `"split.max_rows": 20000` و`"train.epochs": 1` ثم `RUN_NAME = "smoke-1"` (اسم جديد حتى لا يختلط بتشغيل حقيقي).

In [ ]:
# ═══════════ ١) الكود: من أين يأتي وأين يُحفظ ═══════════
TORCH_REPO = "yuosef772424/nig-timenet-torch"   # مستودع GitHub (خاص: يلزم توكن للقراءة)
TORCH_REF = "main"                              # فرع أو وسم أو SHA كامل (40 حرفاً) يُشغَّل الكود منه
REPO_SOURCE = "auto"       # "auto"  : نسخة Drive إن وُجدت في REPO_DIR وإلا GitHub  |  "github": دائماً نسخة نظيفة من GitHub
                           # "drive" : نسخة Drive فقط، بلا شبكة ولا توكن (يفشل إن لم توجد النسخة)
REPO_DIR = "/content/drive/MyDrive/nig_timenet_torch"   # نسخة المشروع الدائمة على Drive (مستودع git)
TORCH_DIR = "/content/nig-timenet-torch"        # نسخة GitHub المؤقتة على قرص الجهاز (حين يكون المصدر github)
UPDATE_DRIVE_COPY = False  # True = قبل التشغيل اجلب TORCH_REF من GitHub إلى نسخة Drive (يتطلب توكناً)
DISCARD_LOCAL_CHANGES = False   # False = يرفض تحديث نسخة Drive إن عدّلتَ فيها ملفاً (لا يمحو عملك بصمت)؛ True = يتجاهل تعديلاتك
# التوكن (للقراءة فقط: Contents: Read-only): الترتيب = المتغيّر GITHUB_TOKEN ← سرّ Colab ← ملف على Drive ← سؤال بحقل مخفي. لا يُطبع ولا يُكتب في .git/config.
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"            # اسم سرّ Colab (الأيقونة 🔑 ← Add new secret ← فعّل Notebook access)
TOKEN_PATH = "/content/drive/MyDrive/crypto/secrets/github_token.txt"   # احتياطي على Drive (أي من يصل إلى Drive يقرؤه)
SAVE_TOKEN_TO_DRIVE = True   # True = التوكن الذي تكتبه يدوياً يُحفظ في TOKEN_PATH؛ توكن سرّ Colab لا يُنسخ إلى Drive أبداً
# ═══════════ ٢) البيئة والجهاز ═══════════
USE_ENV_CACHE = True       # True = بيئة بايثون مخزّنة على Drive (venv مع --system-site-packages: torch الخاص بـ Colab يبقى كما هو)
ENV_CACHE_DIR = "/content/drive/MyDrive/nig_timenet_env"   # مجلد الـ tar: ملف واحد env-<مفتاح>.tar، المفتاح = بايثون + torch + المتطلبات
VENV_DIR = "/content/nig_venv"                  # أين تُفكّ البيئة على الجهاز (نفس المسار الذي بُنيت فيه)
EXTRA_MODULES = (("pyarrow", "pyarrow"),)       # تُثبَّت فقط حين USE_ENV_CACHE = False: (اسم الاستيراد، اسم pip)
EXPECT_DEVICE = "cuda"     # "cuda" | "cpu": الجهاز المتوقَّع؛ تشغيل cuda على جهاز CPU يفشل قبل أي عمل (القاعدة E1)
WORKDIR = "/content/nig_runner"                 # مجلد عمل مؤقت (سجلات، أدوات git)؛ لا يُستخدم /tmp أبداً (القاعدة E2)
DRIVE_RUNS = "/content/drive/MyDrive/nig_timenet_runs"   # جذر التشغيلات الدائمة على Drive (وفيه setup.json)
# ═══════════ ٣) التشغيل: الاسم والمكان والاستعادة ═══════════
RUN_NAME = "notebook-main"      # اسم مجلد هذا التشغيل؛ نفس الاسم = استئناف من نقطة الحفظ؛ اسم جديد = تشغيل جديد
WORK_ON = "drive"          # "drive": يعمل مباشرة على Drive (آمن من الانقطاع، أبطأ)  |  "local": على قرص الجهاز (أسرع) ثم يُنسخ في القسم ١٢
LOCAL_RUNS = "/content/nig_runs"                # جذر العمل حين WORK_ON = "local"
RESTORE_FROM = ""          # "" = لا شيء؛ اسم تشغيل سابق تحت DRIVE_RUNS (أو مسار كامل) يُنسخ إلى مجلد هذا التشغيل قبل البدء (للاستئناف منه)
OVERWRITE_ON_RESTORE = False    # False = يرفض الاستعادة إن اختلف ملف موجود؛ True = يستبدله
PUSH_TO_DRIVE = True       # True = القسم ١٢ ينسخ مجلد التشغيل إلى Drive (لا شيء يُنسخ حين WORK_ON = "drive")
ARTIFACT = "/content/drive/MyDrive/crypto_model/torch_artifacts/preprocessing_output_1"   # ملف البيانات (عقد artifact، القاعدة D1)

# ═══════════ ٤) البطارية (قسم «بطارية مجلد منتهٍ» فقط؛ بطارية التشغيل نفسه من evaluation.battery أدناه) ═══════════
BATTERY_FOLDER = ""        # مجلد تشغيل منتهٍ (فيه battery_table_val/test.parquet) لإعادة البطارية منه؛ "" = تخطَّ القسم ١٣
BATTERY_COST = 0.0008      # التكلفة لكل جهة (الصفقة الكاملة تدفعها مرتين)
BATTERY_VARIANTS = 1       # عدد المتغيّرات التي جُرّبت حتى الآن: كلما زاد اتسعت فواصل الثقة (تصحيح التجارب المتعددة)
# ═══════════ ٤ب) مفاتيح التجربة (اختيارية؛ الافتراضيات = السلوك السابق حرفياً) ═══════════
FX1_BUNDLE = False         # True = يضيف إلى model.overrides المفتاحين bn_warmup وorder_detach_close (إصلاحات FX1، القرار D-022)؛ False = كما كان
FX1_NIG_SCALE = None       # None = معطّل؛ 0.01 = رؤوس NIG تعمل بوحدات التسمية (D-022)؛ يعمل وحده أو مع FX1_BUNDLE
KEEP_LAST_WEIGHTS = False  # True = بعد التدريب يبقى النموذج على أوزان آخر حقبة (restore_best_weights = False)؛ False = أفضل حقبة (الافتراضي)
LOAD_WEIGHTS = "train"     # "train" = درّب أو أكمل (الافتراضي) | "best" أو "last" = حمّل أوزان تشغيل منتهٍ (من RESTORE_FROM) بلا تدريب؛ لا يُجمَع مع KEEP_LAST_WEIGHTS
# ═══════════ ٥) RunSettings كاملاً: كل حقل بافتراضيه الحالي (المعنى | القيم المسموحة | الأثر) ═══════════
# ما ليس هنا: run_dir (يُشتق من RUN_NAME وWORK_ON) وdata.name (= ARTIFACT أعلاه).
CHANGES = {
    "code_sha": "",                  # الـ commit المسجَّل في هوية التشغيل؛ "" = HEAD الحالي لنسخة الكود (الأفضل: اتركه فارغاً)
    # ── البيانات (data_io/loading.py): كيف يُقرأ الملف ──
    "data.directory": ".",           # مجلد الملف؛ يُتجاهل حين ARTIFACT مسار كامل (وهو كذلك هنا)
    "data.format": "auto",           # "auto" (الأحدث بين .dataset و.pkl.gz) | "pkl.gz" | "npy_dir"
    "data.mmap": True,               # True = المصفوفات الكبيرة تبقى على القرص وتُقرأ صفحاتها عند الحاجة (رام أقل)؛ False = حمّل الكل
    "data.local_dir": "auto",        # "auto" = انسخ إلى قرص الجهاز إن كان الملف تحت /content/drive (قراءة memmap من Drive بطيئة)؛ "" = لا تنسخ؛ أو مسار
    "data.allow_unchecked": False,   # False = لا يُحمَّل إلا «عقد» سليم (القاعدة D1: الملف يثبت مصدره والعملات المحجوبة)؛ True = تخطَّ الفحص (خطر)
    "data.verify_hashes": False,     # True = أعد حساب بصمة كل ملف (قراءة كاملة مرة؛ أبطأ، أضمن)
    "data.expect_base_tf": None,     # None = اقبل أي فريم؛ "4h" مثلاً = ارفض ملفاً فريمه الأساسي مختلف (يمنع خلط 1h بـ 4h)
    # ── التقسيم الزمني (workflow/settings.py::SplitSettings) ──
    "split.train_end": "",           # نهاية التدريب (ISO مثل "2025-06-24")؛ يُعطيان معاً أو لا يُعطيان (train_end وval_end)
    "split.val_end": "",             # نهاية التحقق؛ بعدها الاختبار. إن تُركا فارغين: تواريخ الملف نفسه، وإلا النسب أدناه
    "split.train_share": 0.80,       # نسبة التدريب من النوافذ بالزمن (0 < … ومجموعها مع val_share < 1) حين لا تواريخ
    "split.val_share": 0.10,         # نسبة التحقق؛ الباقي (≈10%) اختبار
    "split.embargo_hours": 0.0,      # فجوة عزل إضافية بالساعات بين الأقسام؛ 0 = يكفي العزل التلقائي بنهاية التسمية (label_end)
    "split.max_rows": None,          # None = كل الصفوف؛ رقم (مثل 20000) = وضع smoke: حد أقصى للصفوف في كل قسم بخطوة متساوية
    "split.exclude_coins": [],       # عملات تُحذف صفوفها من التدريب والتحقق والاختبار معاً (بعد التقسيم الزمني، فلا تتحرك التواريخ)؛ مثل ["BTCUSDT", "ETH"]؛ اسم غير موجود في الملف = خطأ؛ [] = كل العملات
    # ── النموذج (model/architectures.py): معمارية مسجَّلة + تعديلاتها ──
    "model.architecture": "nig_timenet_v2",   # اسم المعمارية في السجل ARCHITECTURES؛ اسم غير مسجَّل = خطأ
    "model.overrides": {             # فوق الإعداد الافتراضي للمعمارية: هنا إعداد «مقاومة الحفظ» (ANTI_MEMORIZATION_CONFIG) وهو الافتراضي
        "d_model": 64,               # عرض التمثيل الداخلي (قنوات)؛ أكبر = سعة أكبر وحفظ أسهل
        "num_layers": 2,             # عدد طبقات المُرمِّز
        "head_hidden": 64,           # عرض الطبقة المخفية في رؤوس الانحدار (NIG)
        "class_head_hidden": 32,     # عرض الطبقة المخفية في رؤوس التصنيف
        "dropout": 0.25,             # إسقاط عشوائي أثناء التدريب فقط (تنظيم)
        "input_clip": 4.0,           # قصّ قيم المدخل بعد التطبيع إلى ±4 (يحدّ أثر القيم الشاذة)
        "stats_mode": "symlog",      # طريقة إحصاءات تطبيع النافذة: "symlog" هنا ("full" في الإعداد الأصلي)
        "input_noise_std": 0.1,      # ضجيج غاوسي على المدخل أثناء التدريب فقط
        "feature_dropout": 0.1,      # إسقاط قنوات ميزات كاملة أثناء التدريب
        "linear_path_l2": 0.001,     # عقوبة L2 على المسار الخطي (يمنعه من حمل الإشارة وحده)
        "level_passthrough": True,   # تمرير مستوى السعر الأخير إلى الرؤوس مباشرة
        "level_norm": "batch",       # تطبيع ذلك المستوى: "batch" | None
        "encoder": "gru",            # المُرمِّز المسجَّل (ENCODERS): "gru" هنا
    },
    # ── التدريب (workflow/settings.py::TrainSettings) ──
    "train.epochs": 5,               # عدد الحقب (≥ 1)؛ التشغيل المرجعي أفضل حقبة فيه الأولى (النموذج يحفظ بعدها)
    "train.batch_size": 256,         # حجم الدفعة (≥ 1)؛ نفاد ذاكرة GPU ⇒ قلّله
    "train.seed": 0,                 # بذرة العشوائية (تهيئة الأوزان، الخلط، الإسقاط)؛ مقارنة بين ذراعين تحتاج بذرتين على الأقل (C4)
    "train.lr": 0.001,               # معدّل التعلّم الابتدائي (> 0)
    "train.weight_decay": 0.0001,    # اضمحلال الأوزان (≥ 0)؛ ملاحظة: 1e-4 × lr 1e-3 أثره ضئيل (H-M6)
    "train.label_smoothing": 0.1,    # تنعيم تسميات رؤوس التصنيف (0 ≤ … < 1)
    "train.selection_metric": "val_raw_loss",   # مقياس اختيار أفضل حقبة: "val_raw_loss" | "val_pinball_{high,low,close}_reg" | "val_auc_close_class" (اتجاه high/low غير قابل للاختيار)
    "train.early_stop.patience": 15,           # الحقب بلا تحسّن قبل الإيقاف المبكر (≥ 1)
    "train.early_stop.min_delta": 0.0001,      # أقل تحسّن يُعدّ تحسّناً (≥ 0)
    "train.early_stop.smoothing": "window",    # تنعيم المقياس قبل المقارنة: "none" | "window" (متوسط آخر window حقب) | "ema"
    "train.early_stop.window": 3,              # عرض النافذة حين smoothing = "window"
    "train.early_stop.ema_beta": 0.7,           # معامل EMA حين smoothing = "ema" (0 ≤ … < 1)
    "train.early_stop.restore_best_weights": True,   # True = يعود النموذج لأوزان أفضل حقبة بعد التدريب (التقارير كلها على أفضل أوزان)
    "train.trace_representative": False,   # True = أثر المدرّب (حصص التدرّج لكل رأس) على مجموعة الدفعات التمثيلية (16 دفعة من ≥ 32 عملة) بدل أول صفوف التدريب؛ للتسجيل فقط (D-019)
    "train.grad_clip_norm": None,    # قصّ معيار التدرّج (None = بلا قصّ؛ TF: 1.0) (D-021)
    "train.ema_decay": None,         # متوسط أسّي للأوزان مع إحماء TF (None = بلا EMA؛ TF: 0.999) (D-021)
    "train.init": "torch",           # تهيئة الأوزان: "torch" | "keras" (D-021)
    "train.weight_decay_groups": "all_params",   # اضمحلال الأوزان على: "all_params" | "exclude_norm_bias" (D-021)
    "train.uncertainty_weighting": True,   # أوزان Kendall لرؤوس التصنيف (D-021)
    "train.class_loss_weight": 1.0,  # وزن خسارة كل رأس تصنيف (≥ 0) (D-021)
    "train.lambda_reg_end": 0.05,    # القيمة النهائية لمنظِّم أدلة NIG (≥ 0) (D-021)
    "train.lambda_calib_end": 0.1,   # القيمة النهائية لحد معايرة الثقة (≥ 0) (D-021)
    "train.beta_nll": 0.0,           # beta-NLL: وزن NLL بتباين التوقّع (0 = معطّل) (D-022)
    "train.beta_nll_normalize": False,   # قسمة أوزان beta-NLL على متوسطها في الدفعة (D-022)
    "train.device": "auto",          # "auto" = cuda إن وُجد وإلا cpu | "cuda" | "cpu"
    # ── التقييم (workflow/settings.py::EvalSettings) ──
    "evaluation.chunk": 20000,       # صفوف لكل دفعة تنبؤ عند التقييم (قسم كبير يجب أن يتسع على الجهاز)
    "evaluation.health_max_rows": 100000,   # أقصى صفوف من val لتقرير الصحة (عيّنة بخطوة متساوية)
    "evaluation.battery.enabled": True,     # False = لا بطارية في التشغيل (يبقى تقرير الاختبار)
    "evaluation.battery.cost_per_side": 0.0008,   # التكلفة لكل جهة؛ الصفقة الكاملة = ضعفها. قرار المالك: 0.0008
    "evaluation.battery.n_variants": 1,     # عدد المتغيّرات المجرَّبة حتى الآن (يوسّع فواصل الثقة)
}

## ٢) تحميل الكود — GitHub ↔ نسخة المشروع على Drive

المستودع خاص، ولذلك يلزم توكن GitHub **للقراءة فقط** (fine-grained، صلاحية `Contents: Read-only` على هذا المستودع). مصدر الكود تحدّده `REPO_SOURCE`:

| `REPO_SOURCE` | ماذا يحدث | متى تستعمله |
|---|---|---|
| `"auto"` (الافتراضي) | يستعمل **نسخة Drive** في `REPO_DIR` إن وُجدت (بلا شبكة ولا توكن)، وإلا يستنسخ GitHub نسخة نظيفة في `TORCH_DIR` | الاستعمال اليومي |
| `"github"` | دائماً نسخة نظيفة من GitHub (`rm -rf` ثم `git clone`، فلا تبقى بقايا جلسة سابقة) عند `TORCH_REF` | لتضمن أن كل الحسابات والجلسات تشغّل الكود نفسه بالضبط |
| `"drive"` | نسخة Drive فقط؛ يفشل برسالة واضحة إن لم توجد | تشغيل بلا إنترنت أو من نسخة عدّلتها بنفسك |

**التزامن في الاتجاهين:**
1. **GitHub ← Drive:** اجعل `UPDATE_DRIVE_COPY = True` فتُجلَب `TORCH_REF` من GitHub إلى نسخة Drive (تُستنسخ أول مرة، وتُحدَّث بعدها: `fetch` ثم `checkout`).
   **يرفض التحديث** (`LocalEditsError`) إن كانت في نسخة Drive ملفات معدَّلة أو commit محلي؛ انسخ عملك أولاً أو اضبط `DISCARD_LOCAL_CHANGES = True` لتجاهله عن قصد.
2. **Drive → التشغيل:** `REPO_SOURCE = "drive"` (أو `"auto"` مع وجود النسخة) يشغّل الكود من نسخة Drive كما هي.
3. **نتائج التشغيل ← Drive** و**استعادة تشغيل سابق ← الجهاز:** القسم ٥ والقسم ١٢ أدناه.

> **لا `git push` إلى GitHub من هذا الدفتر.** التوكن للقراءة فقط عن قصد. التعديلات التي تجريها داخل نسخة Drive تبقى في Drive؛ لنقلها إلى GitHub انسخها إلى جهازك وادفعها من هناك (سؤال مفتوح للمالك في وصف الـ PR).

**التوكن:** يُقرأ بالترتيب من المتغيّر `GITHUB_TOKEN`، ثم سرّ Colab، ثم الملف `TOKEN_PATH` على Drive، ثم يُسأل مرة واحدة بحقل مخفي (ويُحفظ في الملف إن كان `SAVE_TOKEN_TO_DRIVE`).
يُتحقَّق منه مع GitHub قبل الاستعمال، **ولا يُطبع ولا يظهر في رسائل الأخطاء ولا يُكتب في `.git/config`** (يُمرَّر لـ git عبر `GIT_ASKPASS` في بيئة العملية الفرعية فقط).
في وضع `"drive"` بلا تحديث لا يُطلب توكن أصلاً.

**الخلية الأولى لا يمكن أن تكون استيراداً عادياً:** لا شيء من الحزمة موجود على الجهاز بعد، فتقرأ ملفاً واحداً بلا اعتمادات (`colab/bootstrap.py`) من نسخة Drive أو من GitHub وتنفّذه.

In [ ]:
import getpass, os, urllib.error, urllib.request
from google.colab import drive
drive.mount("/content/drive")
boot_file = os.path.join(REPO_DIR, "src", "nig_timenet_torch", "colab", "bootstrap.py")
use_drive = REPO_SOURCE == "drive" or (REPO_SOURCE == "auto" and os.path.exists(boot_file))
token, token_source = "", ""
if not use_drive:        # نحتاج GitHub لتنزيل bootstrap.py: ابحث عن توكن دون سؤال أولاً
    token, token_source = os.environ.get("GITHUB_TOKEN", ""), "env"
    if not token:
        try:
            from google.colab import userdata
            token, token_source = userdata.get(GITHUB_TOKEN_SECRET) or "", "secret"
        except Exception:
            token = ""
    if not token and os.path.exists(TOKEN_PATH):
        token, token_source = open(TOKEN_PATH, encoding="utf-8").read().strip(), "file"
    if not token:
        token, token_source = getpass.getpass("GitHub token (not shown): ").strip(), "typed"

In [ ]:
if use_drive:
    boot_source = open(boot_file, encoding="utf-8").read()      # من Drive: بلا شبكة
else:
    url = f"https://api.github.com/repos/{TORCH_REPO}/contents/src/nig_timenet_torch/colab/bootstrap.py?ref={TORCH_REF}"
    req = urllib.request.Request(url, headers={"Authorization": "Bearer " + token, "User-Agent": "nig-colab",
                                               "Accept": "application/vnd.github.raw"})
    try:
        boot_source = urllib.request.urlopen(req, timeout=60).read().decode()
    except urllib.error.HTTPError as e:
        raise SystemExit(f"GitHub answered HTTP {e.code}: the token is missing, expired or cannot read {TORCH_REPO}. "
                         f"Delete {TOKEN_PATH} and run this cell again.")
boot = {}
exec(compile(boot_source, "bootstrap.py", "exec"), boot)

In [ ]:
needs_github = (not use_drive) or UPDATE_DRIVE_COPY
try:
    from google.colab import userdata
    secret_getter = userdata.get
except ImportError:
    secret_getter = None
if needs_github:         # يتحقق من التوكن مع GitHub؛ توكن سرّ Colab لا يُنسخ إلى Drive أبداً
    preset = token or boot["find_token"](GITHUB_TOKEN_SECRET, secret_getter=secret_getter)
    token = boot["acquire_token"](TOKEN_PATH, TORCH_REPO, lambda: getpass.getpass("GitHub token (not shown): "),
                                  preset=preset, save=SAVE_TOKEN_TO_DRIVE and token_source == "typed")
code = boot["load_code"](REPO_SOURCE, TORCH_REPO, TORCH_REF, REPO_DIR, TORCH_DIR, WORKDIR,
                         token if needs_github else "", UPDATE_DRIVE_COPY, DISCARD_LOCAL_CHANGES)
print("الكود يعمل من:", code["resolved"], code["dir"], "| commit", code["sha"][:10])

## ٣) البيئة المخزّنة على Drive (تُوفّر دقائق في كل جلسة جديدة)

جهاز Colab الجديد يعيد تثبيت الحزم نفسها كل مرة. هنا تُبنى البيئة **مرة واحدة** وتُحفظ كملف **tar واحد** على Drive، وتُفكّ في الجلسات التالية خلال ثوانٍ:

* بيئة افتراضية `python -m venv --system-site-packages`: ترى `torch` الجاهز في Colab (بنسخة CUDA الصحيحة) ولا تُنزّل غيره. ملف قيد (`constraints`) يثبّت نسخة torch الحالية فلا يستطيع أي طلب استبدالها.
* **مفتاح التخزين** = بصمة (بايثون + نسخة torch المركّبة + أسطر المتطلبات من `pyproject.toml`). أي تغيير في أحدها ⇒ مفتاح جديد ⇒ تُبنى البيئة من جديد ويُحذف الـ tar القديم.
* الـ tar يُكتب باسم مؤقت ثم يُعاد تسميته، فلا يبقى ملف ناقص إن انقطعت الجلسة؛ وملف تالف يُكتشف فتُعاد البناء تلقائياً.

**فخ:** إن كانت الحزم `numpy` / `pandas` / `pyarrow` مستورَدة بالفعل في الـ kernel (Colab يستوردها غالباً) فلن تُستبدَل نسخها حتى تعيد تشغيل بيئة التشغيل. الخلية تطبع تحذيراً بأسماء ما استُورد مسبقاً؛
عندها `Runtime ▸ Restart session` ثم شغّل من أول الدفتر (البيئة صارت مخزّنة فلن تُبنى ثانيةً).
`USE_ENV_CACHE = False` يعيد السلوك القديم: `pip install --no-deps -e` فقط مع `EXTRA_MODULES`.

ثم تتحقق الخلية التالية من **الجهاز** وتكتب `setup.json` (commit الكود، نسخ الحزم، اسم GPU) في `WORKDIR` و`DRIVE_RUNS`؛ تشغيل `cuda` على جهاز CPU يفشل هنا قبل أي عمل (القاعدة E1).

In [ ]:
import sys
sys.path.insert(0, os.path.join(code["dir"], "src"))        # وحدة env_cache من الكود الذي حمّلناه
if USE_ENV_CACHE:
    from nig_timenet_torch.colab.env_cache import activate_env, ensure_env
    env = ensure_env(os.path.join(code["dir"], "pyproject.toml"), VENV_DIR, ENV_CACHE_DIR)
    active = activate_env(VENV_DIR)
    if active["already_imported"]:
        print("⚠️ استُورد مسبقاً:", active["already_imported"], "— أعد تشغيل الجلسة ثم شغّل من أول الدفتر")
install = boot["install_package"](code["dir"], modules=() if USE_ENV_CACHE else EXTRA_MODULES)

In [ ]:
from nig_timenet_torch.colab.session import check_environment
setup = check_environment(EXPECT_DEVICE, {"torch": code["sha"]}, install, WORKDIR, DRIVE_RUNS)

## ٤) مجلد التشغيل — أين يعمل، والاستعادة من Drive

`run_paths` تحدّد مكان مجلدَي التشغيل من `RUN_NAME` و`WORK_ON`:

* `…/<RUN_NAME>/train`: نقاط الحفظ وحالة الاستئناف وتقرير الصحة. إن وُجدت فيه نقطة حفظ **يستأنف التدريب منها** (إعادة تشغيل الدفتر بنفس `RUN_NAME` تكمل ما انقطع).
* `…/<RUN_NAME>/out`: التقارير والملخّص وجداول البطارية.

`RESTORE_FROM` ينسخ مجلد تشغيل سابق من Drive إلى مجلد هذا التشغيل **قبل** أي خطوة (مثلاً `RESTORE_FROM = "baseline-001-s0-r2"` ثم `RUN_NAME = "continue-1"`).
يرفض استبدال ملف مختلف إلا مع `OVERWRITE_ON_RESTORE = True`، ويتأكد أن المصدر مجلد تشغيل فعلاً (فيه `train/` أو `out/`).

In [ ]:
from nig_timenet_torch.colab.drive_sync import restore_run, run_paths
paths = run_paths(RUN_NAME, DRIVE_RUNS, LOCAL_RUNS, WORK_ON)
restored = restore_run(RESTORE_FROM, paths, overwrite=OVERWRITE_ON_RESTORE)
print("مجلد التدريب:", paths.run_dir, "\nمجلد المخرجات:", paths.out_dir)

## ٥) كائن الإعدادات `RunSettings`

يُبنى من الافتراضيات ثم تُطبَّق عليه `CHANGES` (وملف البيانات `ARTIFACT`). النتيجة كائن **صارم وغير قابل للتعديل**: أي مفتاح غير معروف أو قيمة خارج المسموح يرفعه خطأ هنا، قبل أن يُبنى أي شيء.
اطبعه وراجعه: هذا ما سيُسجَّل في هوية التشغيل (بصمة الإعدادات) ويُقارَن عند الاستئناف.

In [ ]:
from nig_timenet_torch.core.config import updated
from nig_timenet_torch.workflow.settings import RunSettings
if LOAD_WEIGHTS not in ("train", "best", "last"):
    raise ValueError(f"LOAD_WEIGHTS must be 'train', 'best' or 'last', got {LOAD_WEIGHTS!r}")
if LOAD_WEIGHTS != "train" and KEEP_LAST_WEIGHTS:
    raise ValueError("KEEP_LAST_WEIGHTS changes the run identity: leave it False when LOAD_WEIGHTS reads a finished run")
settings = updated(RunSettings(run_dir=paths.run_dir), {**CHANGES, "data.name": ARTIFACT})
extra = {**({"bn_warmup": True, "order_detach_close": True} if FX1_BUNDLE else {}),
         **({"nig_scale": FX1_NIG_SCALE} if FX1_NIG_SCALE is not None else {})}
overlay = {"model.overrides": {**settings.model.overrides, **extra}} if extra else {}
if KEEP_LAST_WEIGHTS:
    overlay["train.early_stop.restore_best_weights"] = False
settings = updated(settings, overlay) if overlay else settings
print(settings)


## ٦) البيانات والتقسيم

**ماذا تفعل `prepare`:** (١) تحمّل الملف **بشرط أن يكون «عقداً» سليماً** (القاعدة D1: يسجّل مصدره، قائمة العملات، العملات المحجوبة المستبعدة، وأن أحدث شمعة قبل ختم الزمن)؛ ملف بلا هذه الحقول يُرفض.
(٢) تقرأ حقائق الملف (وضع الهدف `reg_target_mode`، مفتاح المدخل، حالة الترميز) وتفشل مبكراً إن غابت. (٣) تقسّم **بالزمن** إلى train / val / test مع **عزل** (purge): لا تدخل نافذةٌ في التدريب إن امتدّت تسميتها إلى فترة التحقق.

**كيف تقرأ المخرج:**
* **أعداد النوافذ** لكل قسم: يجب أن يكون train الأكبر بفارق، وval وtest غير فارغين. قسم فارغ = تواريخ التقسيم خاطئة.
* **`train_end` / `val_end`:** تاريخا الفصل. test يلي `val_end` بعد فجوة العزل؛ تأكّد أنها تقع حيث تتوقع (ليست كلها في آخر الملف).
* **شكل `X`:** `(نوافذ، طول النافذة، عدد الميزات)`.
* **`data_fingerprint`:** بصمة ملف البيانات. سجّلها مع أي نتيجة: نتيجتان على بصمتين مختلفتين لا تُقارنان.

**أخطاء شائعة:** `max_rows` مضبوط ⇒ تشغيل تجريبي (لا تقرأ أرقامه كنتيجة)؛ ملف `.dataset` على Drive مباشرة بطيء (لذلك `data.local_dir="auto"` ينسخه إلى القرص المحلي أول مرة).

In [ ]:
from nig_timenet_torch.workflow import run as steps
run = steps.prepare(settings, paths.out_dir)
print({k: len(v) for k, v in run.idx.items()}, "| train_end / val_end:", run.bounds)
print("X train:", tuple(run.x["train"].shape), "| X val:", tuple(run.x["val"].shape), "| targets:", run.targets)
print("data_fingerprint:", run.art.manifest.data_fingerprint, "| max_rows:", settings.split.max_rows)

## ٧) النموذج

`build` يثبّت البذرة (`train.seed`)، ويبني النموذج من المعمارية المسجَّلة (`model.architecture` + `model.overrides`)، ثم المدرّب (المُحسِّن، جدول معدّل التعلّم، أوزان Kendall للخسائر)، ثم **هوية التشغيل**
(بصمة تضمّ الإعدادات والبيانات والكود: بها يُرفض استئناف نقطة حفظ من إعداد مختلف).

**كيف تقرأ المخرج:** اسم المعمارية، **عدد المعاملات** (الإعداد الافتراضي نحو 260 ألف معامل لنافذة وميزات بحجم المعتاد؛ قفزة كبيرة تعني أن `d_model` أو الطبقات تغيّرت)،
وعدد الأهداف (high / low / close، ولكل منها رأس انحدار NIG ورأس تصنيف). قارن عدد المعاملات بعدد نوافذ التدريب الفعّالة: الحفظ يظهر حين تكبر السعة على المعلومة المتاحة (انظر القسم ٨).

In [ ]:
steps.build(run)
n_params = sum(p.numel() for p in run.trainer.model.parameters())
print(run.ms.architecture, "| المعاملات:", f"{n_params:,}", "| الجهاز:", run.device, "| الأهداف:", run.targets)
print(run.ident.fingerprint())

## ٨) التدريب

يطبع سطراً لكل حقبة: `train_loss` و`val_loss` (الخسارة الكلية المرجَّحة) و`val_selection` (مقياس اختيار أفضل حقبة، الافتراضي `val_raw_loss`: مجموع خسائر الرؤوس **بلا ترجيح Kendall** فهو ثابت التعريف ومستقل عن الأوزان — القاعدة C3) ومعدّل التعلّم والزمن.

**كيف تقرأ المخرج:**
* **أفضل حقبة** هي أقل `val_selection`، ويحتفظ النموذج بأوزانها (`restore_best_weights`). إن كانت **الأولى** وبقي `train_loss` ينزل بينما يصعد `val_loss` فهذا **حفظ/فرط تكيّف**: التشغيل المرجعي `baseline-001-s0-r2` كان هكذا، وهو موضوع دراسة M1/M2r.
* **الإيقاف المبكر:** يتوقف التدريب بعد `train.early_stop.patience` حقب بلا تحسّن (بتنعيم النافذة). `stopped_early=True` في الملخّص.
* **الاستئناف:** إن انقطعت الجلسة شغّل الدفتر بنفس `RUN_NAME` فيكمل من آخر نقطة حفظ في `train/`. لا يستأنف إن اختلفت بصمة الإعدادات (يرفض بدل أن يخلط).

**أخطاء شائعة:** `val_loss` أقل من `train_loss` في الحقبة الأولى طبيعي (الإسقاط والضجيج يعملان في التدريب فقط)؛ نفاد ذاكرة GPU ⇒ قلّل `train.batch_size`؛ الخسارة `nan` ⇒ أوقف وأبلغ (لا تتجاهل).
بعد التدريب تُنسَخ نقطة الحفظ إلى Drive فوراً (مع `WORK_ON = "local"`) قبل التقارير، حتى لا يضيع التدريب إن انقطعت الجلسة بعده.
**أوزان آخر حقبة:** بـ`KEEP_LAST_WEIGHTS = True` يبقى النموذج على أوزان آخر حقبة منفَّذة، وتُحسب عليها كل التقارير اللاحقة (`best.pt` يُحفظ كذلك). لقراءة تشغيل منتهٍ بأوزانه بلا تدريب استعمل `LOAD_WEIGHTS = "last"` أو `"best"` مع `RESTORE_FROM`.


In [ ]:
from nig_timenet_torch.workflow.flat_run import epoch_line
if LOAD_WEIGHTS == "train":
    steps.train(run, on_epoch=lambda row: print(epoch_line(row, settings.train.epochs)))
else:
    steps.restore(run, weights=LOAD_WEIGHTS)    # بلا تدريب: أوزان تشغيل منتهٍ من مجلد التدريب (أفضل حقبة أو آخر حقبة)
res = run.result
print("الحقب المنفَّذة:", res.epochs_done, "| أفضل حقبة:", res.best_epoch, "| أفضل قيمة:", res.best_value,
      "| إيقاف مبكر:", res.stopped_early, "| الأوزان المحمَّلة:", "أفضل حقبة" if res.restored_best else "آخر حقبة")


In [ ]:
from nig_timenet_torch.colab.drive_sync import push_run
if PUSH_TO_DRIVE:
    push_run(paths)      # نقطة الحفظ على Drive قبل التقارير (لا يفعل شيئاً حين WORK_ON = "drive")

## ٩) تقرير الصحة (على عيّنة من val)

يفحص النموذج المدرَّب من الداخل (وليس الربح): لكل رأس (high / low / close):

* **rank IC:** ارتباط الرتب بين التوقع والواقع داخل كل طابع زمني ثم المتوسط. **0 = لا مهارة.** الأرقام الصغيرة (0.02–0.05) قد تكون حقيقية في هذا المجال، لكن اقرأها دائماً مقابل خط أساسها.
* **AUC (ثابت / مخلوط):** للتصنيف؛ بجانبه قيمة نموذج ثابت (0.5) ونموذج بتوقعات مخلوطة. high/low «معلومة فقط» (الاتجاه معلَّق).
* **std<move share و coverage80 و sharp ok:** نفس مقاييس T-sharp والتغطية المشروحة في القسم ١٠.
* **حصص أرضية NIG:** نسبة الصفوف التي التصقت فيها معاملات `nu/alpha/beta` بحدّها الأدنى. نسبة عالية = الرأس «انتحر» عند الحد ولا يتعلّم عدم اليقين.
* **فحص الطبقات (layer probe):** هل ما يزال المدخل الخام (`raw_input`) أفضل من تمثيل الطبقة (`trunk_norm`) في التنبؤ بالهدف؟ `delta vs raw` سالبة = الطبقة تُتلف المعلومة، لا تضيف.
* **أثر المدرّب (trainer trace):** الخسائر لكل رأس، أوزان Kendall (يجب ألا تنهار لواحد)، جداول λ.

لاحظ: على بيانات تركيبية صغيرة (اختبارات) تكون الأرقام عشوائية؛ التقرير للتشغيل الحقيقي. يُكتب أيضاً في `health_report.md` بمجلد المخرجات.

In [ ]:
steps.health_step(run)
print(open(paths.out_dir + "/health_report.md", encoding="utf-8").read()[:3000])

## ١٠) تقرير الاختبار (test) — أفضل أوزان، كل صفوف test مرة واحدة

لا يُستعمل test في التدريب ولا في الاختيار أبداً. كل مفتاح في المخرج (`targets.<head>.<قسم>.<مقياس>`) وما يعنيه:

**المهارة `skill` (T-skill):**
* **`rank_ic_by_timestamp`** (rank IC): متوسط ارتباط سبيرمان، عبر العملات داخل كل طابع زمني، بين توقّع النموذج `mu` والقيمة الفعلية. 0 = لا مهارة. **قارنه بـ `…_shuffled_baseline`** (نفس التوقعات بعد خلطها؛ يجب أن يكون ≈ 0).
  تحذير لـ high/low: جزء كبير من مدى الغد مكتوب في مدى آخر شمعة، فالمهارة الحقيقية هي **الزيادة فوق خط أساس التقلب** (تحسبها الخطوة T5r)، لا الرقم الخام.
* **`spearman_per_coin`:** الفكرة نفسها داخل كل عملة عبر الزمن: متوسط الارتباط (`mean`) وحصة العملات الموجبة (`share_positive`). إن كان الـ rank IC موجباً لكن نصف العملات سالب فالإشارة هشّة.
* **`class.auc`:** مساحة تحت المنحنى لرأس التصنيف مقابل `auc_constant_baseline` (=0.5) و`auc_shuffled_baseline`. لـ high/low `information_only=true`: الاتجاه معلَّق ولا يُعدّ دليلاً.

**الثقة `conf` (T-conf):** هل الثقة التي يعلنها النموذج تتنبأ بخطئه؟
* **`decile_error_spearman`:** نقسّم الصفوف داخل كل عملة إلى أعشار بحسب الثقة ونربط رقم العشر بمتوسط الخطأ المطلق. **المطلوب ≤ −0.8** (ثقة أعلى ⇒ خطأ أقل). صفر أو موجب = الثقة لا تعني شيئاً.
* **`top_minus_bottom_error_gap`:** متوسط الخطأ المطبَّع في أقل أعشار الثقة **ناقص** أعلاها؛ موجب = الثقة تفيد. و**`gap_ci95`** فاصل ثقة 95% (بوتستراب على الصفوف، متفائل لأنه يتجاهل الاعتماد الزمني): يجب أن يكون حدّه الأدنى > 0.
* **`constant_confidence_baseline`** = 0: ثقة ثابتة تسجّل صفراً، فلا يُعدّ رقم قريب من الصفر دليلاً.

**الحدّة `sharp` (T-sharp) والتغطية:**
* **`coverage`:** نسبة الصفوف التي وقع الهدف الفعلي داخل فاصل 80% الذي يعلنه النموذج. **المطلوب 0.75–0.85.** أكبر من ذلك = الفواصل أعرض من اللازم (المرجع 0.88–0.93 في التشغيل r2)، أصغر = متفائل أكثر من اللازم.
* **`share_std_below_move`** (T-sharp): حصة الصفوف التي يكون فيها الانحراف المعياري التنبؤي (بوحدة **السعر**) أقل من **الحركة التي توقّعها الرأس نفسه** `|decode(mu) − last_close|`. **المطلوب ≥ 0.80.**
  `zero_std_baseline_share` دائماً 1.0 (σ = 0 «ينجح» دائماً) ⇒ **اقرأ T-sharp مع coverage دائماً**؛ و`median_std_over_move` وسيط النسبة σ/الحركة (أقل من 1 جيد).
* **`meets_target` / `blocking`:** حكم المقياس. لرأس **close**: `blocking=False` — T-sharp عليه معلوماتي فقط (O-1) لأنه لا يتحقق حتى للتوزيع الحقيقي؛ لا يُعامل هدفاً ولا بوابة.

**أخطاء شائعة:** مهارة موجبة لا تعني ربحاً (انظر القسم ١١)؛ `n_rows` يجب أن يساوي عدد صفوف test كلها (لا عيّنة)؛ لا تقارن رقماً من ملف بيانات بآخر دون بصمة البيانات.

In [ ]:
report = steps.test_step(run)
print("صفوف الاختبار:", report["n_rows"])
print("\n".join(steps.test_lines(report)))

## ١١) بطارية التقييم — هل في هذا ربحٌ بعد التكلفة؟ (T-profit)

تُطبَّق على val وtest بعتبات تُضبط على val وتُنفَّذ على test (فلا تسرّب). **التكلفة 0.0008 لكل جهة (0.0016 للصفقة)** من `evaluation.battery.cost_per_side`. الأقسام بالترتيب:

| القسم | يجيب عن | كيف تقرؤه |
|---|---|---|
| TABLE INTEGRITY | هل الجدول سليم؟ | أي NaN أو عدم تطابق أو تداخل بين VAL وTEST ⇒ **توقّف**: تسرّب أو عطل، لا تقرأ ما بعده |
| SANITY | IC واتجاه الإغلاق وانحياز الصعود | `P(p_dir>0.5)` مقابل `P(y>0)`: فرق كبير = انحياز |
| CONFIDENCE CALIBRATION | هل تزداد الدقة مع ثقة النموذج؟ | شرائح الثقة (من الأقل للأعلى): النموذج الموثوق ترتفع دقته بانتظام؛ **ثبات = الثقة بلا معنى** |
| ANTI-COWARDICE | هل يتنبأ بقيم ≈ 0 احتماءً؟ | `mean|mu|/mean|y|` قريب من 0 = جبان ⇒ شبكة الصفقات بعده بلا معنى |
| TRADING GRID / REALISTIC EXECUTION | صفقات سوقية، ثم أوامر محدّدة عند السعر المتوقَّع | n<50 تعني عيّنة صغيرة؛ `fill%` نسبة تنفيذ الأمر |
| UNCERTAINTY FILTER | هل يتحسّن الأداء حين σ صغيرة؟ | إن لم يتحسّن فالنموذج لا يعرف متى لا يعرف |
| SHOCK REGIMES / WEEKLY STABILITY | الأداء في الأسواق الهادئة مقابل الأعنف، وعبر الأسابيع | ميزة تموت في الخُمس الأعنف أو تعيش في أسبوع واحد ليست ميزة |
| **T-PROFIT (close / limit / sigma)** | **الربح بعد التكلفة** للنموذج **مقابل خطوط أساس** | الجدول: `avg` المتوسط بعد التكلفة، `sharpe`، **`CI95`** (فاصل بوتستراب بكتل يومية). **الربح = النموذج يتفوّق على كل خطوط الأساس *و* فاصله يستثني الصفر** |
| VERDICT | الحكم النهائي | يزن **أفضل** استراتيجية فقط ⇒ اقرأ قسمَي الصدمات والأسابيع قبل أن تصدّقه |

خطوط الأساس في T-PROFIT: «دائماً صعود»، «دائماً هبوط»، اتجاه مخلوط، تنبؤات مخلوطة، إزاحة ثابتة، وتحجيم متساوٍ (الأوزان بمتوسط 1 فنفس التعرّض). **رقم ربح بلا هذه الصفوف وبلا الفاصل ليس دليلاً** (القاعدة C1).
عدد المتغيّرات `n_variants` يوسّع الفواصل: كلما جرّبتَ أكثر زاد احتمال الصدفة، فاكتب عدد ما جرّبته.

### ترتيب القراءة المقترح للنتيجة (قائمة فحص)
1. **سلامة الجدول** (TABLE INTEGRITY): أصفار في كل شيء؟ إن لا ⇒ أصلِح قبل أي استنتاج.
2. **تقرير الاختبار:** هل rank IC لرأس close أكبر من خط أساسه المخلوط، وهل نصف العملات على الأقل موجب (`share_positive`)؟ وهل high/low تتفوق فعلاً على خط أساس التقلب (لا الرقم الخام)؟
3. **الثقة:** `decile_error_spearman` بعيد عن −0.8؟ فالثقة لا تُستعمل لتحجيم الصفقات بعد.
4. **التغطية وT-sharp معاً:** تغطية 0.89–0.93 مع حصة T-sharp عالية تعني فواصل **عريضة** تنجح لأنها عريضة، لا لأنها حادة.
5. **T-PROFIT:** هل يتفوّق النموذج على *كل* صفوف الأساس وهل يستثني فاصله الصفر؟ وهل يصمد في الخُمس الأعنف وعبر الأسابيع؟
6. **اكتب ما جرّبتَ:** `n_variants` (كل ما قارنتَه حتى الآن) ثم `commit` الكود و`data_fingerprint`؛ بذرة واحدة تبقى «نظرة» لا نتيجة (C4).

المخرجات تُكتب في مجلد المخرجات: `battery_report.txt` (كل شيء)، `battery_profit.json` (أقسام الربح والحكم)، و`battery_table_{val,test}.parquet` (تعيد منها القسم ١٣ البطارية بلا نموذج).

In [ ]:
from nig_timenet_torch.workflow.battery_run import render
sections = steps.battery_step(run)
print(render(sections))

## ١٢) الملخّص ثم الدفع إلى Drive

`write_summary` يكتب `baseline_summary.json`: البذرة، الحقب المطلوبة والمنفَّذة، **أفضل حقبة**، زمن الحقبة والخطوة، أعداد النوافذ، تواريخ التقسيم، وضع الهدف، **بصمتا البيانات والهوية**، وتاريخ الحقب كاملاً. هذا الملف هو «بطاقة» التشغيل التي تقرؤها الجلسات اللاحقة.

بعده `push_run` ينسخ مجلد التشغيل (التقارير ونقاط الحفظ) إلى `DRIVE_RUNS/<RUN_NAME>` عبر ملف مؤقت ثم إعادة تسمية (لا ملف ناقص). يتخطى الملفات المطابقة، ويستبدل الأقدم بالأحدث.
مع `WORK_ON = "drive"` لا يفعل شيئاً لأن التشغيل كان على Drive أصلاً. **لا يُنسخ إلى GitHub شيء** (انظر القسم ٢).

| الملف في مجلد المخرجات | ما فيه |
|---|---|
| `baseline_summary.json` | بطاقة التشغيل: الإعدادات الفعلية، أفضل حقبة، البصمات، التاريخ |
| `test_report.json` | كل مقاييس القسم ١٠ لكل رأس (وللعملات منفردةً) |
| `health_report.md` / `.json` | القسم ٩ |
| `battery_report.txt` | كل أقسام البطارية كنص |
| `battery_profit.json` | أقسام T-PROFIT والحكم، مع التكلفة وعدد المتغيّرات |
| `battery_table_val.parquet` / `battery_table_test.parquet` | جداول التنبؤات (مدخل القسم ١٣؛ تُقرأ بأي أداة) |
| `drive_sync.json` (في نسخة Drive) | متى نُسخ المجلد آخر مرة وكم ملفاً |

In [ ]:
summary = steps.write_summary(run)
print(summary["epochs_done"], "حقبة | أفضل حقبة", summary["best_epoch"], "| المخرجات في", paths.out_dir)
if PUSH_TO_DRIVE:
    pushed = push_run(paths)

## ١٣) بطارية مجلد تشغيل منتهٍ (أي نموذج)

يقرأ `battery_table_{val,test}.parquet` من مجلد كتبه تشغيل ويعيد **البطارية كلها** بتكلفة أخرى أو بعدد متغيّرات أكبر. **لا يحتاج نموذجاً ولا ملف بيانات**، ويعمل مع أي نموذج يكتب هذين الجدولين.
اضبط `BATTERY_FOLDER` في خلية الإعدادات (مثلاً مجلد `out` لتشغيل سابق استعدته من Drive). اتركه فارغاً لتخطّي القسم.
استعمله لسؤال «ماذا لو كانت التكلفة 0.0012؟» دون إعادة تدريب: `BATTERY_COST = 0.0012`.

In [ ]:
from nig_timenet_torch.workflow.battery_run import battery_from_folder
if BATTERY_FOLDER:
    print(render(battery_from_folder(BATTERY_FOLDER, cost=BATTERY_COST, n_variants=BATTERY_VARIANTS)))

## أخطاء شائعة وحلولها

| الرسالة / العَرَض | السبب المرجَّح | الحل |
|---|---|---|
| `GitHub answered HTTP 401/404` | توكن ناقص أو منتهٍ أو لا يقرأ المستودع | احذف `TOKEN_PATH` وأعد الخلية؛ أنشئ توكن `Contents: Read-only` على المستودع |
| `LocalEditsError: the Drive copy … has local changes` | عدّلتَ ملفاً (أو أنشأتَ commit) داخل نسخة Drive | انسخ عملك خارجها، أو `DISCARD_LOCAL_CHANGES = True` لتجاهله عن قصد |
| `… exists and is not a git copy` | مجلد `REPO_DIR` موجود لكنه ليس مستودع git (نُسخ يدوياً) | انقله جانباً (لا يُلمَس) أو غيّر `REPO_DIR` |
| `no project copy at …` | `REPO_SOURCE = "drive"` ولا نسخة بعد | شغّل مرة بـ `"github"` و`UPDATE_DRIVE_COPY = True` |
| `DeviceError` في القسم ٤ | طلبتَ `cuda` والجهاز CPU | `Runtime ▸ Change runtime type ▸ T4 GPU`، أو `EXPECT_DEVICE = "cpu"` لتشغيل CPU عن قصد |
| `⚠️ استُورد مسبقاً: ['numpy', …]` | حزم مستوردة قبل تفعيل البيئة المخزّنة | `Runtime ▸ Restart session` ثم من أول الدفتر |
| `contract artifact … refused` | ملف البيانات ليس «عقداً» سليماً (D1) | أعد بناءه بـ `data_pipeline.ipynb` أو صدّره بأداة التصدير؛ لا تستعمل `data.allow_unchecked` إلا لفحص |
| `SyncError: … already holds different files` | الاستعادة ستستبدل ملفاً مختلفاً | اختر `RUN_NAME` جديداً أو `OVERWRITE_ON_RESTORE = True` |
| نفاد ذاكرة GPU | دفعة كبيرة أو `evaluation.chunk` كبير | `train.batch_size` أو `evaluation.chunk` أصغر |
| `ValueError: RunSettings: unknown key 'epoch'` | اسم حقل خاطئ في `CHANGES` | الرسالة تقترح الاسم الصحيح (`train.epochs`) |

## ١٤) أقسام الدفتر القديم (TensorFlow) التي ليس لها مقابل هنا بعد

لا نتظاهر بوجودها. هذه أقسام `main.ipynb` في `crypto-signal-prediction` وما يقابلها في هذا المشروع (الخطوات من `docs/ROADMAP.md`):

| قسم الدفتر القديم | ما يفعل | الحالة هنا |
|---|---|---|
| ٣-ب تغيير الهدف (`target_mode`: return / relative / scaled …) | يعيد حساب الأهداف من الأسعار الخام دون إعادة بناء البيانات | **غير مخطَّط.** وضع الهدف يُحدَّد عند بناء الملف (`data_pipeline.ipynb`) ويُقرأ كحقيقة من الملف؛ لا خطوة له في ROADMAP |
| ٧-ب تقييم انتقائي (دقة ≥ 65% على شريحة واثقة، صفقات 2:1) | هل ترفع الثقة الدقة؟ | **جزئياً:** قسما CONFIDENCE CALIBRATION وUNCERTAINTY FILTER في البطارية؛ المقياس الانتقائي المستقل غير مخطَّط (T6a يضيف مقياس اختيار قائماً على الربح) |
| ٧-ج خط أساس شكل الشمعة (high/low) | هل يضيف النموذج شيئاً فوق آخر شمعة؟ | **T5r**: خط أساس التقلب لرأسي high/low (الزيادة فقط تُعدّ مهارة) + صفوف الأساس في البطارية |
| ٧-د التحقق المتكامل `run_full_verification` | جدول حكم واحد | **D1 / D2** (طبيب النموذج: تفعيلات، رؤوس، إسناد، بطارية) |
| ٧-هـ تجربة التحكّم بالتسميات المخلوطة | هل «تتعلّم» الإشارة من تسميات عشوائية؟ | **G1** (بيئة E2: خلط عام للتسميات) و**G2** (أدوات البوابات)، ثم **G3r** على الإعداد المرجعي |
| ٧-و محفظة محايدة للسوق | هل يتحوّل التفوّق النسبي إلى ربح؟ | **جزئياً:** صفوف الربح في البطارية؛ التحقق الصارم (تكاليف واقعية، بيانات قادمة) **«Later 3»** عند طلب المالك فقط |
| ٧-ز مقاومة الحفظ: تدقيق التطبيع وفجوة التعميم | هل يحفظ النموذج؟ | الإعداد نفسه هو الافتراضي؛ الدراسة في **M1** (الجزء على CPU) و**M2r** (أذرع GPU قصيرة)؛ مقياس فجوة train/val في **TR1** |
| ٧-ح نموذج اللوحة عبر العملات (panel) | عيّنة = يوم كامل بكل عملاته | **مؤجَّل** («Later 1»: slot panel) |
| ٧-ط تشخيص النموذج (صحة، طبقات، مسجّل التدريب) | فحص داخلي | **منجز جزئياً:** القسم ٩ أعلاه؛ العمق في **D1 / D2** |
| ٧-ي السعة مقابل العيّنات الفعّالة | هل الحفظ بسبب سعة كبيرة؟ | **M1** (عدّادات H-M1: طوابع مميزة، ارتباط ذاتي، عدد العيّنات الفعّال) |
| معايرة الثقة بعد التدريب (isotonic / platt) | تعديل الفواصل على val | **T6b** البند 5 (`eval/calibrator.py`) |
| أطر متعددة (`model_tfs=("1h","4h")`) | سياق 4h | **غير مخطَّط** (لا خطوة في ROADMAP؛ الملف يعلن فريمه الأساسي ويرفض `data.expect_base_tf` المخالف) |
| تقسيم walk-forward (`rolling_splits`) | استقرار عبر طيّات | **F2** (النصف المتدحرج من I4) |
| ٨ اختبار التوصيل الذاتي (`run_wiring_selftest`) | بيانات تركيبية | **لا يلزم:** هذا الدفتر نفسه يُشغَّل في اختبار آلي على بيانات تجريبية في كل CI (`tests/notebooks/`) |